# Bước 3 trên Kaggle — Phạm Văn Hoàng Anh Tú (2A202602507)

So sánh phương pháp suy luận trên **val** cho cấu hình cuối F01 (không huấn luyện lại, **không đụng test**),
đo độ trễ p50/p95/p99 trên GPU T4 (warmup 10, synchronize, 100 lần đo), rồi chốt phương pháp cho chung kết
theo quy tắc ghi sẵn trong `code/buoc3.py`.

**Trước khi chạy:** bấm **Add Input** (khung bên phải) → tab **Your Work** → chọn notebook đã chạy
`kaggle_buoc2b.ipynb` (có `best.pt` của F01) → **Add**.
Kết quả nhỏ được gói vào `/kaggle/working/kaggle_buoc3.zip`.

In [ ]:
# Ô K1 — kiểm tra GPU, cài thư viện, in phiên bản
import os, sys, platform, subprocess
LOCAL_TEST = os.environ.get("LAB_LOCAL_TEST") == "1"   # chỉ Claude dùng khi chạy thử trên CPU; trên Kaggle luôn False
if not LOCAL_TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm", "openpyxl"], check=True)
import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert LOCAL_TEST or torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

In [ ]:
# Ô K2 — lấy code từ GitHub (nhánh có code đã chạy thử)
ROOT = os.environ.get("LAB_ROOT", "/kaggle")
WORK = f"{ROOT}/working"
REPO = f"{WORK}/repo"
BRANCH = "claude/project-thread-mdpq02"
if not LOCAL_TEST and not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/tubepvhat1604-bot/K4-DAY02-PhamVanHoangAnhTu-2A202602507.git", REPO], check=True)
CODE_DIR = f"{REPO}/submissions/2A202602507_pham_van_hoang_anh_tu/code"
for p in (REPO, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
OUT = f"{WORK}/sub"                      # giống cấu trúc thư mục bài nộp: curves/, predictions/, logs/
RUNS_DIR = f"{WORK}/deepweeds_runs"      # checkpoint + log từng run
for d in ("curves", "predictions", "logs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
os.chdir(OUT)
import train, dataset
from pathlib import Path
from train import Config, run
print("code:", CODE_DIR)

In [ ]:
# Ô K3 — tải ảnh (Zenodo) + CSV fold 0 (GitHub tác giả), kiểm tra MD5, giải nén ra đĩa tạm
import hashlib, zipfile, urllib.request
DATA_DIR   = os.environ.get("LAB_DATA", "/tmp/data")
LABELS_DIR = f"{DATA_DIR}/labels"
ZIP_LOCAL  = f"{DATA_DIR}/images.zip"
ZIP_URL    = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5    = "b7b30f96d466fba86016aa5a26606e0f"
os.makedirs(LABELS_DIR, exist_ok=True)

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not LOCAL_TEST and not os.path.exists(f"{DATA_DIR}/.images_ok"):
    if not os.path.exists(ZIP_LOCAL):
        print("Tải images.zip từ Zenodo...")
        urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    got = md5sum(ZIP_LOCAL)
    assert got == ZIP_MD5, f"MD5 sai: {got}"
    print("MD5 đúng:", got)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f"{DATA_DIR}/images")
    open(f"{DATA_DIR}/.images_ok", "w").close()

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}.csv", f"{LABELS_DIR}/{name}.csv")

IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA_DIR}/images")), key=lambda r: len(os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh .jpg")
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
dataset.check_split(train_df, val_df, test_df, IMAGES_DIR,
                    expected_total=None if LOCAL_TEST else dataset.TOTAL_IMAGES, verbose=False)
print("Kiểm tra chia dữ liệu: ĐẠT")

In [ ]:
# Ô K4 — tìm checkpoint F01 trong Input (output của notebook kaggle_buoc2b)
import glob
INPUT_ROOT = os.environ.get("LAB_INPUT", f"{ROOT}/input")
hits = sorted(glob.glob(f"{INPUT_ROOT}/**/deepweeds_runs/F01/seed0/best.pt", recursive=True))
assert hits, "Chưa thấy F01/seed0/best.pt: bấm Add Input → Your Work → notebook kaggle_buoc2b → Add, rồi chạy lại"
IN_RUNS = str(Path(hits[0]).parents[2])
print("Checkpoint F01 lấy từ:", IN_RUNS, "|", sorted(os.listdir(f"{IN_RUNS}/F01")))

In [ ]:
# Ô K5 — Bước 3: I00-I08 trên val (F01 seed 0; ensemble/soup dùng 3 seed), độ trễ, chốt phương pháp
import buoc3
res, lat, choice = buoc3.run_buoc3(IN_RUNS, IMAGES_DIR, LABELS_DIR, OUT,
                                   latency_iters=5 if LOCAL_TEST else 100,
                                   bn_demo_pretrained=not LOCAL_TEST, num_workers=0 if LOCAL_TEST else 2)
pd.set_option("display.width", 250)
cols = ["method", "views", "aggregate", "input_size", "val_macro_f1", "delta_vs_I00", "val_macro_f1_mean_3seed",
        "val_top1", "val_ece", "lat_b1_p50_ms", "lat_b1_p95_ms", "lat_b1_p99_ms", "cost_vs_I00"]
print(res[cols].round(4).to_string(index=False))
print(lat[["label", "gpu", "dtype", "batch", "img_size", "fused_bn", "p50", "p95", "p99", "images_per_s"]].round(2).to_string(index=False))
print("Chốt cho chung kết:", choice)

In [ ]:
# Ô K6 — gói kết quả Bước 3 thành kaggle_buoc3.zip
import zipfile
zp = Path(WORK) / "kaggle_buoc3.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("inference_val.csv", "latency.csv", "bn_fusion_check.json", "buoc3_choice.json"):
        z.write(Path(OUT, "logs", name), Path("sub", "logs", name))
    z.write(Path(OUT, "curves", "buoc3_tradeoff.png"), Path("sub", "curves", "buoc3_tradeoff.png"))
print(zp, f"{zp.stat().st_size / 1e3:.0f} KB")